# Computational Theory - Jamie Walsh

## Contents

- [Problem 1: Representing SHA-256 data](#problem-1-representing-sha-256-data)

In [60]:
import numpy as np 
import sys

## Problem 1: Representing SHA-256 data

Before beginning any work on implementing SHA-256, this section will cover how we represent SHA-256's inputs, outputs and intermediate data in Python, and why these representations are appropriate.

### 32-bit words

Why 32 bits? SHA-256 requires 32-bit words, as specified in [FIPS 180-4 Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13). Within that detail there's a constraint which is important for how we represent this data: values are limited to 32 bits and overflow if this limited is exceeded.

Unlike "lower-level" languages like C++, where a 32-bit integer comes as a built-in `uint32_t`, Python's `int` does not behave this way.

We need to use a data type that won't exceed 32 bits. To test this, we can take the max value and add 1. Python's plain `int` has no clean way to enforce this limit; however, [Numpy's `np.iinfo`](https://stackoverflow.com/questions/23189506/maximum-allowed-value-for-a-numpy-data-type#:~:text=min_value%20%3D%20np.iinfo(im.dtype).min%0Amax_value%20%3D%20np.iinfo(im.dtype).max) provides a solution. 


In [61]:
limit = np.iinfo(np.uint32).max

print(f"int: {limit + 1}") # plain int: no limit, keeps growing
print(f"np.uint32: {np.uint32(limit) + 1}") # np.uint32: wraps back to 0 

int: 4294967296
np.uint32: 0


/var/folders/3q/zzb1pzp9655bb6h57vz921yh0000gn/T/ipykernel_72319/982339944.py:4: RuntimeWarning: overflow encountered in scalar add
  print(f"np.uint32: {np.uint32(limit) + 1}") # np.uint32: wraps back to 0


This confirms we need `np.uint32` as it wraps the 32-bit limit while Python's plain `int` does not, which FIPS 180-4 requires.

### Sequences of 32-bit words
Also specified in [FIPS 180-4 Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13), SHA-256 represents each 512-bit message block as a sequence of sixteen 32-bit words. To represent this, we use a NumPy array over a standard list, as [Numpy's documentation](https://numpy.org/doc/stable/user/whatisnumpy.html) states arrays are more efficient when every element is the same type, which applies here as every element is a `uint32`.

We can verify this claim in code:


In [ ]:
u32 = np.uint32 # alias for NumPy's unsigned 32-bit integer

numpy_array = np.array([1,2,3,4,5], dtype=u32)
python_list = [u32(x) for x in [1,2,3,4,5]]

print(f"Numpy Array Size: {numpy_array.nbytes}") # array's total memory
print(f"Python List Size: {sys.getsizeof(python_list) + sum(sys.getsizeof(x) for x in python_list)}") # lists total memory: size of pointers + size of every object they point to

print("numpy array:")
%timeit numpy_array + 1

print("python list:")
%timeit [x + 1 for x in python_list]


Numpy Array Size: 20
Python List Size: 260
numpy array:
478 ns ± 1.47 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
python list:
247 ns ± 6.79 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
